# Challenge 22 — Leaderboard Push Notebook

This notebook is the interactive companion for:
`/Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/run_leaderboard_push.py`

The script is the single source of truth for feature building, OOF training, blending, and submission generation.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

BASE_DIR = Path('/Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22')
SCRIPT = BASE_DIR / 'run_leaderboard_push.py'

assert SCRIPT.exists(), f'Missing runner script: {SCRIPT}'
print('Runner script:', SCRIPT)

## 1) Fast Baseline Check

Runs only the anchor weighted Ridge model and produces one submission variant.

In [ ]:
cmd = [
    'python3', str(SCRIPT),
    '--stage', 'baseline',
    '--n-splits', '5',
    '--time-budget-min', '180',
    '--seed', '42',
    '--max-submit-pack', '1',
    '--use-transductive', 'true',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 2) Train + OOF Artifacts

Runs the full model stack with gating and writes:
- `cv_scores.csv`
- `model_registry.json`
- `oof_predictions.parquet`

In [ ]:
cmd = [
    'python3', str(SCRIPT),
    '--stage', 'train',
    '--n-splits', '5',
    '--time-budget-min', '180',
    '--seed', '42',
    '--use-transductive', 'true',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 3) Blend Search

Runs coarse-to-fine blend search and calibration checks.

In [ ]:
cmd = [
    'python3', str(SCRIPT),
    '--stage', 'blend',
    '--n-splits', '5',
    '--time-budget-min', '180',
    '--seed', '42',
    '--use-transductive', 'true',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 4) Submission Pack (5 variants)

Generates variant files under:
`/Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/submissions/`

and writes `submission_manifest.csv` in the latest run folder.

In [ ]:
cmd = [
    'python3', str(SCRIPT),
    '--stage', 'submit_pack',
    '--n-splits', '5',
    '--time-budget-min', '180',
    '--seed', '42',
    '--max-submit-pack', '5',
    '--use-transductive', 'true',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 5) Inspect Latest Run Artifacts

In [ ]:
runs_root = BASE_DIR / 'runs_push'
latest_run = sorted([p for p in runs_root.iterdir() if p.is_dir()])[-1]
print('Latest run:', latest_run)

manifest_path = latest_run / 'submission_manifest.csv'
if manifest_path.exists():
    display(pd.read_csv(manifest_path))
else:
    print('No submission_manifest.csv in latest run.')